In [4]:
import os

In [5]:
os.chdir("../")

In [6]:
%pwd

'd:\\Github Projects\\Pratice'

Entity

In [22]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelTrainerConfig:
    root_dir: Path
    train_data_path: Path
    test_data_path: Path
    model_name: str
    alpha: float
    l1_ratio: float
    target_column: str


Configuration Manager

In [23]:
import sys
from pathlib import Path

from src.datascience.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH, SCHEMA_FILE_PATH
from src.datascience.utils.common import read_yaml,create_directories
# from src.datascience.entity.config_entity import DataIngestionConfig, DataTransformationConfig, DataValidationConfig, ModelTrainerConfig

class ConfigurationManager:
    def __init__(self,config_filepath=CONFIG_FILE_PATH,
                 schema_filepath=SCHEMA_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH):
        
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.model_trainer.root_dir])

    def get_model_trainer_config(self)->ModelTrainerConfig:
        config = self.config.model_trainer
        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
                root_dir=config.root_dir,
                train_data_path=config.train_data_path,
                test_data_path=config.test_data_path,
                model_name=config.model_name,
                alpha=self.params.ElasticNet.alpha,
                l1_ratio=self.params.ElasticNet.l1_ratio,
                target_column=next(iter(self.schema.TARGET_COLUMN))
        )

        return model_trainer_config

components

In [24]:
from src.datascience.constants import *
from src.datascience.utils.common import read_yaml, create_directories

In [25]:
from sklearn.linear_model import ElasticNet
import pandas as pd
import joblib
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train(self):
        train_data = pd.read_csv(self.config.train_data_path)
        test_data = pd.read_csv(self.config.test_data_path)

        train_x = train_data.drop(columns=[self.config.target_column])
        train_y = train_data[self.config.target_column]
        test_x = test_data.drop(columns=[self.config.target_column])
        test_y = test_data[self.config.target_column]

        lr = ElasticNet(alpha=self.config.alpha, l1_ratio=self.config.l1_ratio)
        lr.fit(train_x, train_y)
        test_pred = lr.predict(test_x)

        joblib.dump(lr, os.path.join(self.config.root_dir,self.config.model_name))


In [27]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer_config = ModelTrainer(config=model_trainer_config)
    model_trainer_config.train()
except Exception as e:
    print(f"Error occurred while initializing ConfigurationManger: {e}")

[2026-08-28 13:22:55,909: INFO: common: Directory created: artifacts/model_trainer]
[2026-08-28 13:22:55,911: INFO: common: Directory created: artifacts/model_trainer]


pipeline